In [1]:
import os, re, json, hashlib, time, warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple
import requests
import fitz
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import gradio as gr
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
print('All imports successful ✅')

All imports successful ✅


In [2]:
GROQ_API_KEY = ""  
groq_client = Groq(api_key=GROQ_API_KEY)

MAX_PROMPT_CHARS = 7000

def truncate_chunk(text, max_chars=800):
    return text[:max_chars] + " [...]" if len(text) > max_chars else text

def llm_generate(prompt):
    if len(prompt) > MAX_PROMPT_CHARS:
        cutoff = prompt.rfind("\n", 0, MAX_PROMPT_CHARS - 200)
        if cutoff == -1:
            cutoff = MAX_PROMPT_CHARS - 200
        prompt = prompt[:cutoff] + "\n\n[Context truncated]\n\nANSWER:"
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=600
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Error: {e}]"

DOCS_DIR      = Path('fca_docs')
DB_DIR        = Path('chroma_db')
MANIFEST_FILE = Path('doc_manifest.json')
DOCS_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)

CHUNK_SIZE     = 1500  
CHUNK_OVERLAP  = 200
TOP_K_RETRIEVE = 10
TOP_K_RERANK   = 3

print(llm_generate("Say hello in one word"))
print('Config done ✅')

Hello
Config done ✅


In [3]:
DOCUMENT_SOURCES = [
    {
        'name': 'Payment Services Regulations 2017',
        'url': 'https://www.legislation.gov.uk/uksi/2017/752/pdfs/uksi_20170752_en.pdf',
        'filename': 'payment_services_2017.pdf'
    },
    {
        'name': 'Electronic Money Regulations 2011',
        'url': 'https://www.legislation.gov.uk/uksi/2011/99/pdfs/uksi_20110099_en.pdf',
        'filename': 'electronic_money_2011.pdf'
    },
    {
        'name': 'Financial Services and Markets Act 2000',
        'url': 'https://www.legislation.gov.uk/ukpga/2000/8/pdfs/ukpga_20000008_en.pdf',
        'filename': 'fsma_2000.pdf'
    }
]

def load_manifest():
    if MANIFEST_FILE.exists():
        with open(MANIFEST_FILE) as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    with open(MANIFEST_FILE, 'w') as f:
        json.dump(manifest, f, indent=2)

def fetch_documents():
    manifest = load_manifest()
    updated_files = []
    headers = {'User-Agent': 'Mozilla/5.0 (academic research project)'}

    for source in DOCUMENT_SOURCES:
        filepath = DOCS_DIR / source['filename']
        print(f"\n📄 Checking: {source['name']}")
        try:
            response = requests.get(source['url'], headers=headers, timeout=30)
            response.raise_for_status()
            new_hash = hashlib.md5(response.content).hexdigest()
            old_hash = manifest.get(source['filename'], {}).get('hash', '')

            if new_hash != old_hash or not filepath.exists():
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                manifest[source['filename']] = {
                    'hash': new_hash,
                    'name': source['name'],
                    'url': source['url'],
                    'last_updated': datetime.now().isoformat(),
                    'size_kb': len(response.content) // 1024
                }
                updated_files.append(source['filename'])
                print(f"   ✅ Downloaded ({len(response.content)//1024} KB)")
            else:
                print(f"   ⏭️  No changes — skipping")
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            if filepath.exists():
                print(f"   📁 Using cached version")

    save_manifest(manifest)
    print(f"\n✅ Done. {len(updated_files)} document(s) updated.")
    return updated_files

updated_docs = fetch_documents()


📄 Checking: Payment Services Regulations 2017
   ⏭️  No changes — skipping

📄 Checking: Electronic Money Regulations 2011
   ⏭️  No changes — skipping

📄 Checking: Financial Services and Markets Act 2000
   ⏭️  No changes — skipping

✅ Done. 0 document(s) updated.


In [4]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ''
    for page in doc:
        page_text = page.get_text()
        if len(page_text.strip()) < 50:
            blocks = page.get_text("blocks")
            page_text = '\n'.join([b[4] for b in blocks if b[4].strip()])
        text += page_text + '\n'
    doc.close()
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def semantic_chunk(text, source_name, filename):
    # Split on double newlines (paragraph boundaries)
    paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if len(p.strip()) > 80]
    chunks = []
    current_chunk = ''
    chunk_idx = 0

    for para in paragraphs:
        if current_chunk and (len(current_chunk) + len(para) > CHUNK_SIZE):
            chunks.append({
                'text': current_chunk.strip(),
                'source': source_name,
                'filename': filename,
                'chunk_id': f"{filename}_{chunk_idx}",
                'chunk_index': chunk_idx
            })
            overlap_text = current_chunk[-CHUNK_OVERLAP:] if len(current_chunk) > CHUNK_OVERLAP else current_chunk
            current_chunk = overlap_text + ' ' + para
            chunk_idx += 1
        else:
            current_chunk += (' ' + para) if current_chunk else para

    if current_chunk.strip():
        chunks.append({
            'text': current_chunk.strip(),
            'source': source_name,
            'filename': filename,
            'chunk_id': f"{filename}_{chunk_idx}",
            'chunk_index': chunk_idx
        })
    return chunks

all_chunks = []
for source in DOCUMENT_SOURCES:
    filepath = DOCS_DIR / source['filename']
    if not filepath.exists():
        print(f"⚠️  {source['filename']} not found, skipping")
        continue
    print(f"Processing: {source['name']}...")
    text = extract_text_from_pdf(filepath)
    chunks = semantic_chunk(text, source['name'], source['filename'])
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

print(f"\n✅ Total chunks: {len(all_chunks)}")

Processing: Payment Services Regulations 2017...
  → 140 chunks
Processing: Electronic Money Regulations 2011...
  → 66 chunks
Processing: Financial Services and Markets Act 2000...
  → 321 chunks

✅ Total chunks: 527


In [5]:
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded ✅")

chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
collection_name = 'fca_regulations'

# Force rebuild so new chunks are indexed
try:
    chroma_client.delete_collection(collection_name)
    print("Old collection cleared")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={'hnsw:space': 'cosine'}
)

print(f"Embedding {len(all_chunks)} chunks...")
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    texts = [c['text'] for c in batch]
    embeddings = embedder.encode(texts, show_progress_bar=False).tolist()
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=[c['chunk_id'] for c in batch],
        metadatas=[{'source': c['source'], 'filename': c['filename'],
                   'chunk_index': c['chunk_index']} for c in batch]
    )
    print(f"  Batch {i//batch_size + 1} done")

print(f"✅ Vector store ready: {collection.count()} chunks")

# Build BM25 index over all chunks for hybrid retrieval
bm25_corpus = [c['text'].lower().split() for c in all_chunks]
bm25_index = BM25Okapi(bm25_corpus)
bm25 = bm25_index
print("BM25 index ready ✅")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded ✅
Old collection cleared
Embedding 527 chunks...
  Batch 1 done
  Batch 2 done
  Batch 3 done
  Batch 4 done
  Batch 5 done
  Batch 6 done
✅ Vector store ready: 527 chunks
BM25 index ready ✅


In [6]:
print("Loading reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Reranker ready ✅")

Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker ready ✅


In [7]:
def hybrid_retrieve(query, top_k=10, alpha=0.5):
    """
    Combine vector search + BM25 keyword search.
    alpha=1.0 → pure vector, alpha=0.0 → pure BM25, 0.5 → equal blend.
    """
    # --- Vector search ---
    q_emb = embedder.encode([query]).tolist()[0]
    vec_results = collection.query(
        query_embeddings=[q_emb],
        n_results=min(top_k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )
    vec_chunks    = vec_results['documents'][0]
    vec_metas     = vec_results['metadatas'][0]
    vec_distances = vec_results['distances'][0]

    # Convert cosine distance to similarity score, normalise to [0,1]
    vec_scores = [1 - d for d in vec_distances]
    max_vs = max(vec_scores) if vec_scores else 1
    vec_scores = [s / max_vs for s in vec_scores]

    # --- BM25 search ---
    bm25_raw = bm25_index.get_scores(query.lower().split())
    top_bm25_idx = np.argsort(bm25_raw)[::-1][:top_k]
    max_bs = bm25_raw[top_bm25_idx[0]] if bm25_raw[top_bm25_idx[0]] > 0 else 1

    # --- Merge scores (dedup by chunk text) ---
    seen = {}
    for chunk, meta, vs in zip(vec_chunks, vec_metas, vec_scores):
        seen[chunk] = {'chunk': chunk, 'meta': meta, 'score': alpha * vs}
    for idx in top_bm25_idx:
        chunk = all_chunks[idx]['text']
        bs = float(bm25_raw[idx]) / max_bs
        if chunk in seen:
            seen[chunk]['score'] += (1 - alpha) * bs
        else:
            seen[chunk] = {
                'chunk': chunk,
                'meta': {'source': all_chunks[idx]['source'],
                         'filename': all_chunks[idx]['filename'],
                         'chunk_index': all_chunks[idx]['chunk_index']},
                'score': (1 - alpha) * bs
            }
    merged = sorted(seen.values(), key=lambda x: x['score'], reverse=True)[:top_k]
    return [m['chunk'] for m in merged], [m['meta'] for m in merged]


def baseline_rag(query):
    query_embedding = embedder.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(TOP_K_RETRIEVE, collection.count()),  # fixed: was TOP_K_RERANK
        include=['documents', 'metadatas', 'distances']
    )
    chunks    = results['documents'][0][:TOP_K_RERANK]
    metadatas = results['metadatas'][0][:TOP_K_RERANK]

    context = '\n\n---\n\n'.join([
        f"[Source: {m['source']}]\n{truncate_chunk(chunk)}"
        for chunk, m in zip(chunks, metadatas)
    ])

    prompt = f"""You are a regulatory compliance assistant for FinTech Product Owners.
Answer using ONLY the provided context. If not in context, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    return {
        'answer': llm_generate(prompt),
        'chunks': chunks,
        'sources': [m['source'] for m in metadatas],
        'pipeline': 'baseline'
    }

print("Baseline pipeline ready ✅")

Baseline pipeline ready ✅


In [8]:
def hyde_generate_hypothetical(query):
    prompt = f"""You are a UK financial regulation expert.
Write a short hypothetical passage (3-4 sentences) that would appear in an official
UK financial regulation document answering this question.
Write in formal regulatory language.

Question: {query}

Regulatory passage:"""
    return llm_generate(prompt)

def hybrid_retrieve(query, hypothetical, top_k=10):
    """Combine vector search on HyDE doc + BM25 keyword search on original query."""
    # Vector search using hypothetical document embedding
    hyp_embedding = embedder.encode([hypothetical]).tolist()[0]
    vec_results = collection.query(
        query_embeddings=[hyp_embedding],
        n_results=min(top_k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )
    vec_chunks   = vec_results['documents'][0]
    vec_metas    = vec_results['metadatas'][0]
    vec_scores   = [1 - d for d in vec_results['distances'][0]]  # cosine sim

    # BM25 keyword search on original query
    query_tokens = query.lower().split()
    bm25_scores_all = bm25.get_scores(query_tokens)
    bm25_top_idx = np.argsort(bm25_scores_all)[::-1][:top_k]

    # Merge: collect all candidate chunk texts with scores
    candidates = {}
    for chunk, meta, score in zip(vec_chunks, vec_metas, vec_scores):
        cid = meta['filename'] + '_' + str(meta['chunk_index'])
        candidates[cid] = {'chunk': chunk, 'meta': meta,
                           'vec_score': score, 'bm25_score': 0.0}

    for idx in bm25_top_idx:
        c = all_chunks[idx]
        cid = c['filename'] + '_' + str(c['chunk_index'])
        bm25_norm = float(bm25_scores_all[idx]) / (max(bm25_scores_all) + 1e-9)
        if cid in candidates:
            candidates[cid]['bm25_score'] = bm25_norm
        else:
            candidates[cid] = {'chunk': c['text'], 'meta': c,
                               'vec_score': 0.0, 'bm25_score': bm25_norm}

    # Combine scores (alpha=0.6 vector, 0.4 BM25)
    alpha = 0.6
    for cid in candidates:
        candidates[cid]['hybrid_score'] = (
            alpha * candidates[cid]['vec_score'] +
            (1 - alpha) * candidates[cid]['bm25_score']
        )

    sorted_cands = sorted(candidates.values(), key=lambda x: x['hybrid_score'], reverse=True)[:top_k]
    return [c['chunk'] for c in sorted_cands], [c['meta'] for c in sorted_cands]

def enhanced_rag(query):
    # Step 1: HyDE — generate hypothetical answer for better embedding
    hypothetical = hyde_generate_hypothetical(query)

    # Step 2: Hybrid retrieval (vector + BM25)
    chunks, metadatas = hybrid_retrieve(query, hypothetical, top_k=TOP_K_RETRIEVE)

    # Step 3: Rerank with cross-encoder
    pairs = [[query, chunk] for chunk in chunks]
    scores = reranker.predict(pairs)
    ranked_idx = np.argsort(scores)[::-1][:TOP_K_RERANK]

    top_chunks = [chunks[i] for i in ranked_idx]
    top_metas  = [metadatas[i] for i in ranked_idx]
    top_scores = [float(scores[i]) for i in ranked_idx]

    context = '\n\n---\n\n'.join([
        f"[Source: {m['source']} | Relevance: {s:.2f}]\n{truncate_chunk(chunk)}"
        for chunk, m, s in zip(top_chunks, top_metas, top_scores)
    ])

    prompt = f"""You are a regulatory compliance assistant for FinTech Product Owners.
Answer using ONLY the provided context. Cite the specific regulation and section.
If the answer is not in the context, say exactly that.

CONTEXT:
{context}

QUESTION: {query}

ANSWER (cite sources):"""

    return {
        'answer': llm_generate(prompt),
        'chunks': top_chunks,
        'sources': list(set([m['source'] for m in top_metas])),
        'rerank_scores': top_scores,
        'hypothetical': hypothetical,
        'pipeline': 'enhanced'
    }

print("Enhanced pipeline (HyDE + Hybrid + Reranker) ready ✅")

Enhanced pipeline (HyDE + Hybrid + Reranker) ready ✅


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# ── Test Queries ──────────────────────────────────────────────────────────────
TEST_QUERIES = [
    {'query': 'What is strong customer authentication?',
     'type': 'simple_factual',
     'expected_keywords': ['authentication', 'customer', 'payment', 'identity']},
    {'query': 'What are the capital requirements for payment institutions?',
     'type': 'simple_factual',
     'expected_keywords': ['capital', 'payment institution', 'requirement', 'funds']},
    {'query': 'What are the conditions under which a payment service provider can refuse a transaction?',
     'type': 'deep_context',
     'expected_keywords': ['refuse', 'transaction', 'payment service provider', 'conditions']},
    {'query': 'What information must be provided before a payment transaction?',
     'type': 'deep_context',
     'expected_keywords': ['information', 'payment', 'user', 'transaction']},
    {'query': 'What are the liability rules when a payment transaction is unauthorised?',
     'type': 'deep_context',
     'expected_keywords': ['liability', 'unauthorised', 'payment', 'payer']},
    {'query': 'What are the consent requirements?',
     'type': 'ambiguous',
     'expected_keywords': ['consent', 'payment', 'authorisation']},
    {'query': 'What are the notification obligations?',
     'type': 'ambiguous',
     'expected_keywords': ['notification', 'notify', 'inform', 'obligation']},
    {'query': 'What are the requirements for storing customer payment data?',
     'type': 'po_specific',
     'expected_keywords': ['data', 'payment', 'security']},
    {'query': 'What are the exemptions from strong customer authentication?',
     'type': 'po_specific',
     'expected_keywords': ['exemption', 'authentication', 'low value']},
    {'query': 'What happens if a payment institution becomes insolvent?',
     'type': 'edge_case',
     'expected_keywords': ['insolvency', 'safeguarding', 'funds', 'protection']},
]

# ── Ground Truth Answers ──────────────────────────────────────────────────────
# Written based on actual content of the sourced FCA/legislation documents
GROUND_TRUTH = [
    "Strong customer authentication requires at least two independent elements: knowledge such as a password, possession such as a phone, and inherence such as a fingerprint.",
    "Payment institutions must hold initial capital between 20000 and 125000 euros depending on service type, plus ongoing own funds as a percentage of payment volume.",
    "A payment service provider may refuse a transaction if it is unlawful, technically impossible, or fraudulent, and must notify the payer of the refusal and reasons where possible.",
    "Before a payment transaction the provider must give the payer information on maximum execution time, applicable charges, exchange rates, and the payee identifier.",
    "For unauthorised transactions payer liability is capped at 35 pounds unless the payer acted fraudulently or with gross negligence, after which full liability rests with the payment service provider.",
    "Consent to a payment transaction must be given in the form agreed between payer and provider and may be withdrawn before the transaction becomes irrevocable.",
    "Payment service providers must notify users without undue delay of refused transactions, upcoming changes to framework contracts, and any suspected fraud on the account.",
    "Payment data must be stored with encryption and strict access controls in line with PCI-DSS standards and FCA operational resilience requirements.",
    "Exemptions from strong customer authentication include low value contactless payments under 30 pounds, trusted beneficiary transactions, and recurring payments of identical amounts.",
    "If a payment institution becomes insolvent safeguarded client funds held separately from institution assets are protected and returned to clients ahead of general creditors.",
]

# ── Metric Functions ──────────────────────────────────────────────────────────
def retrieval_precision(chunks, keywords):
    """Fraction of retrieved chunks containing at least one expected keyword."""
    relevant = sum(1 for c in chunks if any(kw.lower() in c.lower() for kw in keywords))
    return relevant / len(chunks) if chunks else 0

def retrieval_recall(chunks, keywords):
    """
    Fraction of ALL relevant chunks in corpus that were retrieved.
    Note: recall is naturally low with 527 chunks and only 3 retrieved —
    this is a known limitation of small top-k retrieval evaluated against
    a large corpus without full manual annotation.
    """
    total_relevant = sum(
        1 for c in all_chunks
        if any(kw.lower() in c['text'].lower() for kw in keywords)
    )
    retrieved_relevant = sum(
        1 for c in chunks
        if any(kw.lower() in c.lower() for kw in keywords)
    )
    return retrieved_relevant / total_relevant if total_relevant > 0 else 0

def answer_completeness(answer, keywords):
    """Fraction of expected keywords appearing in the generated answer."""
    hits = sum(1 for kw in keywords if kw.lower() in answer.lower())
    return hits / len(keywords)

def grounding_score(answer, chunks):
    """
    Measures how grounded the answer is in retrieved chunks.
    Computes fraction of 3-word phrases in answer that appear in context.
    """
    words = answer.lower().split()
    phrases = [' '.join(words[i:i+3]) for i in range(len(words) - 2)]
    if not phrases:
        return 0
    chunks_text = ' '.join(chunks).lower()
    grounded = sum(1 for p in phrases if p in chunks_text)
    return min(grounded / len(phrases), 1.0)

def correctness_score(answer, reference):
    """
    Semantic correctness: cosine similarity between answer and ground truth
    embeddings. Score of 1.0 = identical meaning, 0.0 = unrelated.
    """
    ans_emb = embedder.encode([answer])
    ref_emb = embedder.encode([reference])
    return float(cosine_similarity(ans_emb, ref_emb)[0][0])

# ── Run Evaluation ────────────────────────────────────────────────────────────
print("Running evaluation on 10 queries × 2 pipelines...")
print("="*60)
results = []

for i, test in enumerate(TEST_QUERIES):
    print(f"\nQuery {i+1}/{len(TEST_QUERIES)}: {test['query'][:55]}...")

    b = baseline_rag(test['query'])
    time.sleep(1)
    e = enhanced_rag(test['query'])
    time.sleep(1)

    print(f"  HyDE: {e['hypothetical'][:80]}...")

    results.append({
        'query': test['query'],
        'type':  test['type'],

        # Retrieval metrics
        'baseline_retrieval_precision': retrieval_precision(b['chunks'], test['expected_keywords']),
        'baseline_retrieval_recall':    retrieval_recall(b['chunks'], test['expected_keywords']),
        'enhanced_retrieval_precision': retrieval_precision(e['chunks'], test['expected_keywords']),
        'enhanced_retrieval_recall':    retrieval_recall(e['chunks'], test['expected_keywords']),

        # Generation metrics
        'baseline_answer_completeness': answer_completeness(b['answer'], test['expected_keywords']),
        'baseline_grounding':           grounding_score(b['answer'], b['chunks']),
        'baseline_correctness':         correctness_score(b['answer'], GROUND_TRUTH[i]),
        'enhanced_answer_completeness': answer_completeness(e['answer'], test['expected_keywords']),
        'enhanced_grounding':           grounding_score(e['answer'], e['chunks']),
        'enhanced_correctness':         correctness_score(e['answer'], GROUND_TRUTH[i]),

        # Raw answers (for demo log)
        'baseline_answer': b['answer'],
        'enhanced_answer': e['answer'],
    })

    r = results[-1]
    print(f"  Precision  — Baseline: {r['baseline_retrieval_precision']:.2f} | Enhanced: {r['enhanced_retrieval_precision']:.2f}")
    print(f"  Recall     — Baseline: {r['baseline_retrieval_recall']:.3f} | Enhanced: {r['enhanced_retrieval_recall']:.3f}")
    print(f"  Correctness— Baseline: {r['baseline_correctness']:.2f} | Enhanced: {r['enhanced_correctness']:.2f}")

df = pd.DataFrame(results)

# ── Summary Table ─────────────────────────────────────────────────────────────
metrics_map = {
    'Retrieval Precision':   'retrieval_precision',
    'Retrieval Recall':      'retrieval_recall',
    'Answer Completeness':   'answer_completeness',
    'Grounding Score':       'grounding',
    'Correctness (Semantic)':'correctness',
}

summary = pd.DataFrame({
    'Metric':   list(metrics_map.keys()),
    'Baseline': [df[f'baseline_{v}'].mean() for v in metrics_map.values()],
    'Enhanced': [df[f'enhanced_{v}'].mean() for v in metrics_map.values()],
}).round(3)

summary['Improvement'] = (
    (summary['Enhanced'] - summary['Baseline'])
    / summary['Baseline'].replace(0, 1) * 100
).round(1).astype(str) + '%'

print("\n" + "="*60)
print("OVERALL RESULTS SUMMARY")
print("="*60)
print(summary.to_string(index=False))
print("="*60)

# ── Per Query Type Breakdown ──────────────────────────────────────────────────
print("\n" + "="*60)
print("BREAKDOWN BY QUERY TYPE")
print("="*60)
type_cols = {
    'B_Prec':    'baseline_retrieval_precision',
    'E_Prec':    'enhanced_retrieval_precision',
    'B_Recall':  'baseline_retrieval_recall',
    'E_Recall':  'enhanced_retrieval_recall',
    'B_Correct': 'baseline_correctness',
    'E_Correct': 'enhanced_correctness',
}
type_df = df.groupby('type')[[v for v in type_cols.values()]].mean().round(3)
type_df.columns = list(type_cols.keys())
print(type_df.to_string())
print("="*60)

# ── Note on Recall ────────────────────────────────────────────────────────────
print("""
NOTE ON RECALL:
Recall scores appear low because the metric measures retrieved chunks (3)
against all relevant chunks in the full 527-chunk corpus. Without manual
annotation of all chunks, true recall cannot be computed precisely.
Precision and correctness are more reliable indicators of retrieval quality.
""")

Running evaluation on 10 queries × 2 pipelines...

Query 1/10: What is strong customer authentication?...
  HyDE: "Strong customer authentication (SCA) is a multifactor authentication process, a...
  Precision  — Baseline: 1.00 | Enhanced: 1.00
  Recall     — Baseline: 0.014 | Enhanced: 0.014
  Correctness— Baseline: 0.70 | Enhanced: 0.71

Query 2/10: What are the capital requirements for payment instituti...
  HyDE: "Pursuant to the Payment Services Regulations 2017, payment institutions subject...
  Precision  — Baseline: 1.00 | Enhanced: 1.00
  Recall     — Baseline: 0.010 | Enhanced: 0.010
  Correctness— Baseline: 0.59 | Enhanced: 0.69

Query 3/10: What are the conditions under which a payment service p...
  HyDE: "A payment service provider may refuse to execute a payment transaction where it...
  Precision  — Baseline: 1.00 | Enhanced: 1.00
  Recall     — Baseline: 0.013 | Enhanced: 0.013
  Correctness— Baseline: 0.75 | Enhanced: 0.86

Query 4/10: What information must be provide

In [10]:
DEMO_INDICES = [0, 2, 5, 8, 9]

for i, idx in enumerate(DEMO_INDICES):
    test = TEST_QUERIES[idx]
    b = baseline_rag(test['query'])
    time.sleep(1)
    e = enhanced_rag(test['query'])
    time.sleep(1)

    print(f"\n{'='*65}")
    print(f"DEMO {i+1} | {test['type'].upper()}")
    print(f"{'='*65}")
    print(f"QUERY: {test['query']}")
    print(f"\n💡 HyDE Hypothetical Document:")
    print(f"  {e['hypothetical'][:300]}")
    print(f"\n--- BASELINE ---")
    print(b['answer'][:400])
    print(f"Sources: {b['sources']}")
    print(f"\n--- ENHANCED (HyDE + Hybrid Retrieval + Reranker) ---")
    print(e['answer'][:400])
    print(f"Sources: {e['sources']}")
    print(f"Rerank scores: {[f'{s:.3f}' for s in e['rerank_scores']]}")


DEMO 1 | SIMPLE_FACTUAL
QUERY: What is strong customer authentication?

💡 HyDE Hypothetical Document:
  "Strong Customer Authentication (SCA) is a security measure, as required by the Payment Services Regulations 2017, that is designed to verify the identity of a customer before authorizing a financial or payment transaction. This may involve a combination of two or more of the following elements: kno

--- BASELINE ---
In accordance with regulation 100(1) of the Payment Services Regulations 2017, strong customer authentication is a procedure which allows a payment service provider to verify the identity of a payment service user or the validity of the use of a specific payment instrument, including the use of the user's personalised security credentials.
Sources: ['Payment Services Regulations 2017', 'Payment Services Regulations 2017', 'Payment Services Regulations 2017']

--- ENHANCED (HyDE + Hybrid Retrieval + Reranker) ---
Strong customer authentication is the application of stron

In [11]:
def chat(query, pipeline_choice):
    if not query.strip():
        return 'Please enter a question.', ''
    if pipeline_choice == 'Enhanced (HyDE + Reranker)':
        result = enhanced_rag(query)
        meta = f"**Sources:** {', '.join(result['sources'])}\n\n**Rerank scores:** {[f'{s:.3f}' for s in result['rerank_scores']]}"
    else:
        result = baseline_rag(query)
        meta = f"**Sources:** {', '.join(result['sources'])}"
    return result['answer'], meta

with gr.Blocks(title='RegBot', theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏦 RegBot — FCA Regulatory Assistant for FinTech POs")
    with gr.Row():
        with gr.Column(scale=3):
            q = gr.Textbox(label='Your Question', lines=3,
                           placeholder='e.g. What are SCA requirements?')
            p = gr.Radio(['Baseline RAG', 'Enhanced (HyDE + Reranker)'],
                         value='Enhanced (HyDE + Reranker)', label='Pipeline')
            btn = gr.Button('Ask', variant='primary')
        with gr.Column(scale=4):
            ans = gr.Textbox(label='Answer', lines=10)
            src = gr.Markdown()
    btn.click(chat, [q, p], [ans, src])

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
